This tutorial can be followed through your browser, using Colab [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/main/tutorials/3.3_Inverse_Dynamics.ipynb).

# 3.3 — Inverse Dynamics and Static Optimization

From a measured motion to the muscle activations that could produce it, in two steps:

1. **Inverse dynamics** — the generalized forces $\tau = M(q)\ddot q + c(q, \dot q) - f_\text{passive}$ the model needs to follow the motion. On a floating-base model the free-joint rows of $\tau$ are the wrench the ground must supply.
2. **Static optimization** — per frame, the activations $a$ minimising $\sum a^2$ whose muscle forces (MuJoCo's own force-length-velocity model) reproduce $\tau$. Reserve actuators take what the muscles cannot; optionally, the ground reactions are estimated too.

We use OpenSim's gait2392 walking trial (the same `.mot` file as tutorial 3.5) on MyoLeg (80 muscles). Tutorial 3.4 then drives a forward simulation with **computed muscle control**.

**Prerequisites:** Completed notebook 1.1 · `myosuite` installed · CPU only, under a minute.

*Earlier versions of this tutorial were written by Andrea Prestia.*

In [ ]:
import os; _nb = globals().get("__vsc_ipynb_file__"); _nb and os.chdir(os.path.dirname(_nb))  # VS Code: work relative to this notebook

#  If running in Colab, install myosuite and fetch the tutorial files
if os.getenv("COLAB_RELEASE_TAG"):
    !pip install git+https://github.com/MyoHub/myosuite
    os.environ["MUJOCO_GL"] = "egl"
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/MyoHub/myosuite myosuite
    %cd myosuite
    !git sparse-checkout set tutorials
    %cd tutorials

In [ ]:
import copy
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from myosuite import make_env
from myosuite.utils.video_io import show_video, write_video

files = next(p for p in (Path("files"), Path("tutorials/files")) if p.exists())
sys.path.insert(0, str((files / "3.3").resolve()))
import opensim_gait  # noqa: E402  (maps the OpenSim walk onto MyoLeg, renders it)

**MyoSuite biomechanics modules.** All imports of the inverse-dynamics pipeline live in this one cell.

In [ ]:
# --- MyoSuite biomechanics (inverse dynamics, static optimization) ---
from myosuite.physics.inverse_dynamics import clip_kinematics, inverse_dynamics
from myosuite.physics.static_optimization import StaticOptimizationConfig, static_optimization

## The motion

`opensim_gait.treadmill_walk_to_qpos` maps the OpenSim coordinates onto MyoLeg by name (pelvis → free root, knee sign flipped, knee couplings on their polynomials) and resamples them to 100 Hz. The trial was recorded on a treadmill; adding the belt speed (estimated from the stance foot) gives the same dynamics overground, and the model is lowered so its feet touch the floor.

In [ ]:
env = make_env("myoLegWalk-v0")
model = copy.deepcopy(env.unwrapped.model)  # 80 muscles, free root
env.close()

mot = opensim_gait.read_mot(files / "3.5" / "subject01_walk1_ik.mot")
qpos, belt_speed = opensim_gait.treadmill_walk_to_qpos(model, mot, rate=100.0)
dt = 0.01
body_mass = model.body_mass.sum()
body_weight = body_mass * np.linalg.norm(model.opt.gravity)
print(f"{len(qpos)} frames at {1 / dt:.0f} Hz, belt speed {belt_speed:.2f} m/s, body mass {body_mass:.1f} kg")

## Kinematics

`clip_kinematics` low-pass filters every coordinate (zero-phase Butterworth, 6 Hz as is common for gait) and differentiates twice on the configuration manifold (quaternions included). Accelerations are less reliable within about $1/f_c$ of either end, so we drop those frames from the plots.

In [ ]:
kin = clip_kinematics(model, qpos, dt, lowpass_hz=6.0)
time_s = np.arange(len(qpos)) * dt
keep = slice(20, -20)  # skip 0.2 s at both ends (filter edge effects)


def dof(name: str) -> int:
    """Velocity index of a joint."""
    return int(model.jnt_dofadr[model.joint(name).id])


def qadr(name: str) -> int:
    """Position index of a joint."""
    return int(model.jnt_qposadr[model.joint(name).id])


JOINTS = ["hip_flexion_r", "knee_angle_r", "ankle_angle_r"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, name in zip(axes, JOINTS):
    ax.plot(time_s, np.degrees(qpos[:, qadr(name)]), lw=3, alpha=0.3, label="mapped")
    ax.plot(time_s, np.degrees(kin.qpos[:, qadr(name)]), label="filtered")
    ax.set(title=name, xlabel="time (s)", ylabel="angle (deg)")
axes[0].legend()
fig.tight_layout()
plt.show()

## Inverse dynamics

`inverse_dynamics` returns, per frame, the generalized force actuators, constraints and external loads must apply together. MyoLeg has no actuators on its free root, so its first three rows are the force the ground must exert on the body: the vertical one oscillates around body weight with the familiar double hump. Without a measured ground reaction, the joint rows still lack the load each stance foot carries; static optimization distributes it below.

In [ ]:
t0 = time.time()
tau = inverse_dynamics(model, kin.qpos, kin.qvel, kin.qacc)
print(f"inverse dynamics: {time.time() - t0:.2f} s for {len(tau)} frames")

plt.figure(figsize=(5, 3))
plt.plot(time_s[keep], tau[keep, 2] / body_weight)
plt.axhline(1.0, color="k", lw=0.5)
plt.title("root: vertical force the ground must supply")
plt.xlabel("time (s)")
plt.ylabel("body weights")
plt.show()

## Static optimization

`static_optimization` solves one QP per frame (OSQP, warm-started):

$$\min_{a, r, f}\ \sum_m a_m^2 + \sum_j \left(\frac{r_j}{F_{r}}\right)^2 + \sum_c \left(\frac{f_c}{F_c}\right)^2 \quad \text{s.t.}\quad R^T\left(g \odot a + b\right) + r + J_c^T f + J_\text{eq}^T\lambda = \tau,\quad 0 \le a \le 1,\ f \ge 0$$

- $R$ moment arms, $g$ and $b$ each muscle's active force per unit activation and passive force at the frame's state;
- $r$ reserve actuators on every DoF (on the free root: residual forces);
- with `contact_forces=True`, $f$ are contact forces on MuJoCo's friction pyramid at every foot-floor contact within `contact_distance`, so the ground reactions are **estimated** instead of measured. The model is not scaled to the subject, so its stance feet sit up to a few cm off the floor; a 5 cm reach covers that;
- $\lambda$ the forces of the knee couplings.

In [ ]:
config = StaticOptimizationConfig(contact_forces=True, contact_distance=0.05)
t0 = time.time()
so = static_optimization(model, kin.qpos, kin.qvel, kin.qacc, config=config)
print(f"static optimization: {time.time() - t0:.1f} s, {np.mean(so.status == 'solved'):.0%} of frames solved")
residual = np.sqrt(np.mean(so.reserve[keep, :3] ** 2, axis=0)) / body_weight
print("RMS residual force (x, y, z):", np.round(residual, 3), "body weights")

The estimated vertical ground reaction of each foot, the activations of a few large muscles of the right leg, and what the residuals still carry. Residuals flag where model and data disagree (here mainly the unscaled model); a subject-scaled model shrinks them.

In [ ]:
def foot_force(side: str) -> np.ndarray:
    """Vertical contact force on one foot (heel + toes) [N]."""
    return sum(so.contact_force[:, model.body(f"{b}_{side}").id, 2] for b in ("calcn", "toes"))


MUSCLES = ["glmax1_r", "recfem_r", "vasmed_r", "bflh_r", "gasmed_r", "soleus_r", "tibant_r"]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for side in ("r", "l"):
    axes[0].plot(time_s[keep], foot_force(side)[keep] / body_weight, label=f"foot {side}")
axes[0].plot(time_s[keep], tau[keep, 2] / body_weight, "k:", label="required (ID)")
axes[0].set(title="estimated ground reaction", xlabel="time (s)", ylabel="body weights")
axes[0].legend(fontsize="small")
for name in MUSCLES:
    axes[1].plot(time_s[keep], so.activation[keep, model.actuator(name).id], label=name)
axes[1].set(title="activations (right leg)", xlabel="time (s)", ylim=(0, 1))
axes[1].legend(fontsize="small", ncol=2)
for k, axis in enumerate("xyz"):
    axes[2].plot(time_s[keep], so.reserve[keep, k] / body_weight, label=f"residual {axis}")
axes[2].set(title="root residual force", xlabel="time (s)", ylabel="body weights")
axes[2].legend(fontsize="small")
fig.tight_layout()
plt.show()

With the estimated ground reactions, $\tau$ minus the contact forces' generalized force is the net joint moment the muscles (and reserves) produce. The ankle's plantarflexion moment peaks around 1.5 N m/kg in late stance, as in gait-lab data.

In [ ]:
joint_moment = (tau - so.contact_qfrc) / body_mass
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, name in zip(axes, JOINTS):
    ax.plot(time_s[keep], joint_moment[keep, dof(name)])
    ax.axhline(0.0, color="k", lw=0.5)
    ax.set(title=name, xlabel="time (s)", ylabel="moment (N m / kg)")
fig.tight_layout()
plt.show()

## Video

The filtered motion with the optimised activations: muscles redden as they activate.

In [ ]:
frames = opensim_gait.render_walk(model, kin.qpos[keep], so.activation[keep], every=2)
os.makedirs("videos", exist_ok=True)
output_name = "videos/myoleg_static_optimization.mp4"
write_video(output_name, frames, fps=50)
show_video(output_name)

## Next

- Static optimization predicts no co-contraction and ignores activation dynamics; these activations do not reproduce the motion when played forward. **Tutorial 3.4** closes that loop with computed muscle control.
- Measured ground reactions can replace the estimate: map force-plate wrenches with `mujoco.mj_applyFT` and pass them as `external_qfrc` to `inverse_dynamics` / `static_optimization`.
- For live data, `StaticOptimizer(model).solve(qpos, qvel, qacc)` solves one frame at a time with a warm QP.